In [1]:
import json
import re
from pathlib import Path

stats = []
for path in sorted(Path("../data/processed").glob("*.json")):
	doc = json.loads(path.read_text(encoding="utf-8"))
	tables = [b for b in doc["blocks"] if b["type"] == "table"]

	# header line has no text
	no_header = sum(not re.search(r"\w", t["content"].splitlines()[0]) for t in tables)
	one_row = sum(len(t["content"].splitlines()) <= 3 for t in tables)   # header + separator + 1 row
	no_context = sum(not t["context"] for t in tables)

	stats.append((doc["ticker"], len(tables), no_header, one_row, no_context))

print(f"{'ticker':<6} {'tables':>6} {'no_hdr':>7} {'1row':>6} {'no_ctx':>6}")
for ticker, n, nh, one, nc in sorted(stats, key=lambda s: -s[2] / s[1]):
	print(f"{ticker:<6} {n:>6} {nh / n:>7.0%} {one / n:>6.0%} {nc:>6}")

ticker tables  no_hdr   1row no_ctx
PFE       264     78%    57%      1
GOOGL     179     58%    58%      0
JPM       676     54%    55%      0
XOM       308     54%    58%      1
JNJ       235     54%    55%      0
GS        489     52%    52%      0
BAC       354     52%    52%      0
CVX       290     33%    53%      0
AMZN       90     29%    31%      0
WMT       108     23%    25%      0
CAT       126     19%     6%      0
NKE        86     19%    20%      0
AAPL       60      8%     3%      0
V          92      7%    12%      0
NVDA       64      6%     6%      0
KO        117      5%     7%      0
BA        123      2%     7%      0
UNH        73      1%     7%      0
MSFT       84      1%     1%      0
PG         87      0%     8%      1


In [2]:
# Finds PFE's parsed file without hardcoding the date
pfe_path = next(Path("../data/processed").glob("PFE_10-K_*.json"))
doc = json.loads(pfe_path.read_text(encoding="utf-8"))

small = [
	b for b in doc["blocks"]
	# Keeps tables with at most three lines: header, separator, and one data row
  	if b["type"] == "table" and len(b["content"].splitlines()) <= 3   # one data row
]
print(f"{len(small)} one-row tables\n")

for b in small[:15]:
	print(b["content"].splitlines()[2][:150])   # line 2 = the data row

151 one-row tables

| Washington, D.C. 20549 |
| ☒ | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 |
| ☐ | TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 |
| (State or other jurisdiction of incorporation or organization) | (I.R.S. Employer Identification Number) |
| Securities registered pursuant to Section 12(g) of the Act: None |
| Portions of the Proxy Statement for the 2026 Annual Meeting of Shareholders | Part III |
| TABLE OF CONTENTS |
| DEFINED TERMS |
| Pfizer Inc. | 2025 Form 10-K | i |
| Pfizer Inc. | 2025 Form 10-K | ii |
| Pfizer Inc. | 2025 Form 10-K | iii |
| Pfizer Inc. | 2025 Form 10-K | iv |
| Pfizer Inc. | 2025 Form 10-K | 1 |
| Pfizer Inc. | 2025 Form 10-K | 2 |
| PART I |
